# DS4/GSE278450 — build the merged raw-count AnnData object

This notebook reconstructs the GSE278450 object from the 22 compressed
sample-level files ending in `_RawData.tsv.gz`.

Each file is read as a gene-by-cell table:

- the first column contains gene names;
- the remaining columns contain cell-level raw counts;
- the table is transposed to produce a cell-by-gene matrix;
- the sample identifier is obtained from the filename;
- the sample identifier is appended to every cell barcode to guarantee
  uniqueness across samples.

Files ending in `_Normalized...` are intentionally ignored.

## Expected merged object

- 22 samples
- 91,336 cells
- 36,116 genes

## Default paths

Input:

`data/raw/GSE278450/`

Output:

`data/interim/DS4_GSE278450/GSE278450_merged_rawcounts.h5ad`

Paths can be overridden with `GLIOMA_PROJECT_ROOT`,
`GSE278450_RAW_DIR`, and `GSE278450_MERGED_H5AD`.

In [ ]:
import gc
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE278450_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE278450",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS4_GSE278450"
        / "GSE278450_merged_rawcounts.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS4_GSE278450" / "00a_build"
)

EXPECTED_SAMPLE_IDS = {
    "GSM8546833_BT356",
    "GSM8546834_BT363",
    "GSM8546835_BT364",
    "GSM8546836_BT368",
    "GSM8546837_BT369",
    "GSM8546838_BT373",
    "GSM8546839_BT374",
    "GSM8546840_BT377",
    "GSM8546841_BT380",
    "GSM8546842_BT389A",
    "GSM8546843_BT389",
    "GSM8546844_BT390",
    "GSM8546845_BT396",
    "GSM8546846_BT397",
    "GSM8546847_BT400",
    "GSM8546848_BT402",
    "GSM8546849_BT407",
    "GSM8546850_BT408",
    "GSM8546851_BT409",
    "GSM8546852_BT419",
    "GSM8546853_BT420",
    "GSM8546854_BT426",
}

EXPECTED_N_SAMPLES = 22
EXPECTED_OUTPUT_SHAPE = (91_336, 36_116)

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"GSE278450 RAW directory not found: {RAW_DIR}\n"
        "Extract the *_RawData.tsv.gz files there or set "
        "GSE278450_RAW_DIR."
    )

In [ ]:
FILE_PATTERN = re.compile(r"^(GSM\d+_.+)_RawData\.tsv\.gz$")


def discover_raw_tables(raw_dir: Path) -> list[dict]:
    records = []

    for path in sorted(raw_dir.glob("*_RawData.tsv.gz")):
        match = FILE_PATTERN.match(path.name)
        if match is None:
            raise ValueError(
                f"Unexpected RawData filename: {path.name!r}"
            )
        sample_id = match.group(1)
        records.append(
            {
                "sample_id": sample_id,
                "path": path,
            }
        )

    observed = {record["sample_id"] for record in records}

    if observed != EXPECTED_SAMPLE_IDS:
        missing = sorted(EXPECTED_SAMPLE_IDS - observed)
        extra = sorted(observed - EXPECTED_SAMPLE_IDS)
        raise ValueError(
            "The discovered GSE278450 RawData files do not match the "
            f"historical 22-sample set. Missing={missing}; extra={extra}"
        )

    if len(records) != EXPECTED_N_SAMPLES:
        raise AssertionError(
            f"Expected {EXPECTED_N_SAMPLES} RawData files, "
            f"observed {len(records)}."
        )

    return records


input_records = discover_raw_tables(RAW_DIR)

inventory = pd.DataFrame(
    [
        {
            "sample_id": record["sample_id"],
            "filename": record["path"].name,
            "compressed_size_bytes": record["path"].stat().st_size,
        }
        for record in input_records
    ]
)
inventory.to_csv(
    AUDIT_DIR / "GSE278450_input_file_inventory.tsv",
    sep="\t",
    index=False,
)
inventory

In [ ]:
def numeric_count_array(frame: pd.DataFrame, source: Path) -> np.ndarray:
    if all(pd.api.types.is_numeric_dtype(dtype) for dtype in frame.dtypes):
        values = frame.to_numpy(copy=False)
    else:
        try:
            numeric = frame.apply(pd.to_numeric, errors="raise")
        except Exception as exc:
            raise ValueError(
                f"A non-numeric count was found in {source.name}."
            ) from exc
        values = numeric.to_numpy(copy=False)

    if not np.isfinite(values).all():
        raise ValueError(f"Non-finite count found in {source.name}.")
    if np.any(values < 0):
        raise ValueError(f"Negative count found in {source.name}.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"Non-integer count found in {source.name}.")

    maximum = values.max(initial=0)
    if maximum > np.iinfo(np.int32).max:
        raise OverflowError(
            f"A count in {source.name} exceeds int32: {maximum}"
        )

    return values.astype(np.int32, copy=False)


def read_raw_table(path: Path, sample_id: str) -> ad.AnnData:
    frame = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        index_col=0,
        low_memory=False,
    )

    if frame.shape[0] == 0 or frame.shape[1] == 0:
        raise ValueError(f"{path.name}: parsed table is empty.")

    gene_names = pd.Index(frame.index.astype(str), name="gene")
    cell_barcodes = pd.Index(
        frame.columns.astype(str),
        name="cell_barcode",
    )

    if not gene_names.is_unique:
        duplicate_examples = (
            gene_names[gene_names.duplicated(keep=False)]
            .unique()
            .tolist()[:10]
        )
        raise ValueError(
            f"{path.name}: duplicate gene names were found, including "
            f"{duplicate_examples}. The historical merge requires unique "
            "gene rows within each sample."
        )

    if not cell_barcodes.is_unique:
        duplicate_examples = (
            cell_barcodes[cell_barcodes.duplicated(keep=False)]
            .unique()
            .tolist()[:10]
        )
        raise ValueError(
            f"{path.name}: duplicate cell columns were found, including "
            f"{duplicate_examples}."
        )

    values = numeric_count_array(frame, path)
    X = sp.csr_matrix(values.T, dtype=np.int32)

    cell_ids = pd.Index(
        [f"{barcode}-{sample_id}" for barcode in cell_barcodes],
        name="cell_id",
    )
    obs = pd.DataFrame(index=cell_ids)
    obs["cell_barcode"] = cell_barcodes.to_numpy()
    obs["sample_id"] = sample_id
    obs["sample"] = sample_id
    obs["geo_id"] = "GSE278450"
    obs["core_dataset"] = "DS4_GSE278450"
    obs["source_file"] = path.name

    var = pd.DataFrame(index=gene_names)

    sample = ad.AnnData(X=X, obs=obs, var=var)
    sample.uns["source_file"] = path.name
    sample.uns["source_geo_accession"] = sample_id.split("_", 1)[0]

    del frame, values
    gc.collect()

    return sample

In [ ]:
sample_objects = []
sample_summary_rows = []

for index, record in enumerate(input_records, start=1):
    print(
        f"[{index}/{len(input_records)}] Reading "
        f"{record['path'].name}"
    )

    sample = read_raw_table(
        path=record["path"],
        sample_id=record["sample_id"],
    )

    sample_summary_rows.append(
        {
            "sample_id": record["sample_id"],
            "cells": int(sample.n_obs),
            "genes": int(sample.n_vars),
            "total_counts": int(sample.X.sum()),
            "maximum_count": int(sample.X.max()),
            "source_file": record["path"].name,
        }
    )
    sample_objects.append(sample)
    print(sample)

sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(
    AUDIT_DIR / "GSE278450_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)
sample_summary

In [ ]:
adata_merged = ad.concat(
    sample_objects,
    axis="obs",
    join="outer",
    merge="same",
    index_unique=None,
    fill_value=0,
    pairwise=False,
)

adata_merged.var_names_make_unique()

if not adata_merged.obs_names.is_unique:
    raise AssertionError("Merged cell identifiers are not unique.")

if adata_merged.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "The GSE278450 merge did not reproduce the historical object. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed "
        f"{adata_merged.shape}. Inspect the per-sample summary."
    )

if sp.issparse(adata_merged.X):
    adata_merged.X = adata_merged.X.tocsr().astype(
        np.int32,
        copy=False,
    )
else:
    adata_merged.X = sp.csr_matrix(
        np.asarray(adata_merged.X, dtype=np.int32)
    )

values = adata_merged.X.data
if not np.isfinite(values).all():
    raise AssertionError("Non-finite counts detected after merging.")
if np.any(values < 0):
    raise AssertionError("Negative counts detected after merging.")
if not np.equal(values, np.floor(values)).all():
    raise AssertionError("Non-integer counts detected after merging.")

adata_merged.uns["dataset_id"] = "DS4_GSE278450"
adata_merged.uns["geo_accession"] = "GSE278450"
adata_merged.uns["build_timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)
adata_merged.uns["source_sample_count"] = EXPECTED_N_SAMPLES
adata_merged.uns["feature_join"] = "outer"
adata_merged.uns["input_format"] = (
    "gzip-compressed gene-by-cell RawData TSV tables"
)

build_summary = {
    "dataset_id": "DS4_GSE278450",
    "geo_accession": "GSE278450",
    "n_samples": int(adata_merged.obs["sample_id"].nunique()),
    "n_cells": int(adata_merged.n_obs),
    "n_genes": int(adata_merged.n_vars),
    "matrix_format": type(adata_merged.X).__name__,
    "matrix_dtype": str(adata_merged.X.dtype),
    "total_counts": int(adata_merged.X.sum()),
    "maximum_count": int(adata_merged.X.max()),
    "output_h5ad": str(OUTPUT_H5AD),
}

with open(
    AUDIT_DIR / "GSE278450_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

print(adata_merged)
print(adata_merged.obs["sample_id"].value_counts())
print(build_summary)

In [ ]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_merged.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_OUTPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in (
        "cell_barcode",
        "sample_id",
        "sample",
        "source_file",
    ):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

print("Saved and validated:", OUTPUT_H5AD)